In [ ]:
import os 
from dotenv import load_dotenv,find_dotenv
_ =load_dotenv(find_dotenv())
groq_api_key = os.environ["GROQ_API_KEY"]

In [ ]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader('Data/person.txt')

loaded_data = loader.load()

loaded_data

## Character text splitter
Loopp once for all document and generate chunks

In [ ]:
from langchain_text_splitters import CharacterTextSplitter

text_splitter = CharacterTextSplitter(

    separator = "/n/n",
    chunk_size = 1000,
    chunk_overlap = 200,
    length_function = len,
    is_separator_regex= False,
)

texts = text_splitter.create_documents(loaded_data[0].page_content)
len(texts)

## Recursive character text splitter
Loop 2 times to generate chunks

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

recursive_splitter = RecursiveCharacterTextSplitter(

    chunk_size = 26,
    chunk_overlap = 4
)

text = recursive_splitter.split_text(loaded_data[0].page_content)
text

## Generate Embeddings
Free way is sentece transformer or one can use openai  

In [ ]:
from sentence_transformers import SentenceTransformer

embeding_model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = embeding_model.encode(text)

embeddings

## Vector stores aka Vector Database | ChromaDB,Posgres-pgvector,Pinecone,
stores the embeddings in a very fast searchable database
vector db stores the embeddings in a multi diamentional space so it can cluster the similar data 

In [26]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document

loaded_text = TextLoader('data/data.txt').load()

recursive_splitter = RecursiveCharacterTextSplitter(

    chunk_size = 26,
    chunk_overlap = 4 
)
chunks = recursive_splitter.split_text(loaded_text[0].page_content)

#covert chunk into document object
documents = [Document(page_content=t) for t in chunks]

#chroma db accepts model not the pre-computed embeddings
embedding_model = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')

vector_db = Chroma.from_documents(documents,embedding_model)




## Perform similarity search on chromaDB

In [29]:
question = 'where the food system is connected to?'

answer = vector_db.similarity_search(question)
print(answer[0].page_content)

Food systems are closely


## Vector Store as a Retriver -mostly used
* Find the embeddings that bests answer your question

In [34]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

loaded_text = TextLoader('Data/data.txt').load()

text_splitter = CharacterTextSplitter(chunk_size=1000,chunk_overlap=0)

chunks = text_splitter.split_documents(loaded_text)

embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')

vector_db = FAISS.from_documents(chunks,embeddings)



In [43]:
retriver = vector_db.as_retriever()

answer = retriver.invoke('where the food is connected to')
print(answer[0].page_content)

Urban planning should ensure that green spaces are available to people in every part of the city. Poor communities should not be left with fewer parks or less access to clean air. Trees should be selected according to the local climate, soil conditions, available space, and water requirements. Native plants are often useful because they are adapted to the region and may require less maintenance. Proper planning can also prevent tree roots from damaging roads, buildings, and underground infrastructure.\n\n

Food systems are closely connected to sustainable urban development. Cities depend on long supply chains to bring food from rural areas and other countries. These systems may consume fuel and create packaging waste. Urban farming, community gardens, rooftop agriculture, and local food markets can help shorten the distance between producers and consumers. Growing vegetables locally can also improve food security and provide educational opportunities for children and adults.\n\n
